## DSPy Prompts to Assess Specificity in Questions

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.teleprompt import BootstrapFewShotWithRandomSearch
from dspy.evaluate.evaluate import Evaluate
import json
import random

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=500, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

In [3]:
test_inputs = [
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How big is your organization?",
            "response_categories": []
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How big is your organization, in terms of number of full-time equivalent (FTE) employees?",
            "response_categories": []
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "What is your income?",
            "response_categories": []
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "What was your total household income before taxes in 2023?",
            "response_categories": []
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you do any sports or hobbies involving physical activities, or any exercise, including walking, on a regular basis?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How often do you feel sad?",
            "response_categories": []
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How often have you felt sad during the past week?",
            "response_categories": []
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How good is the economy these days?",
            "response_categories": []
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you agree with the following statement: I seldom abstain from eating meat?",
            "response_categories": [
                {"id": 0, "text": "Agree"},
                {"id": 1, "text": "Disagree"},
            ]
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "How often do you abstain from eating meat?",
            "response_categories": [
                {"id": 0, "text": "Always"},
                {"id": 1, "text": "Often"},
                {"id": 2, "text": "Sometimes"},
                {"id": 3, "text": "Seldom"},
                {"id": 4, "text": "Never"},
            ]
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How many times did you visit the doctor for cough and cold in the past one year?",
            "response_categories": []
        },
        "specificity": "pass"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Is the government’s response appropriate?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "specificity": "fail"
    },
]

In [5]:
tricky_test_inputs = [
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you exercise or play sports regularly?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose raising taxes on the rich?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"},
            ]
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "How often do you visit the doctor for cough and cold?",
            "response_categories": [
                {"id": 0, "text": "Always"},
                {"id": 1, "text": "Often"},
                {"id": 2, "text": "Sometimes"},
                {"id": 3, "text": "Seldom"},
                {"id": 4, "text": "Never"},
            ]
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How important is it that a candidate shares your values?",
            "response_categories": []
        },
        "specificity": "fail"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How important is it that a candidate shares your religious values?",
            "response_categories": []
        },
        "specificity": "pass"
    },
]

In [7]:
# helper function to test out a program on question bank

def test_program(program, test_inputs, verbose=True):

    num_correct = 0

    # rand_int = random.randint(1, 100)

    # running the predictor
    for test_input in test_inputs:
        # stringify the input
        test_input_str = json.dumps(test_input["question"])
        # test_input_str = test_input["main_text"]
        # result = program(question=test_input_str, config=dict(temperature=0.7+0.0001*rand_int))
        result = program(question=test_input_str)
        rationale = result.rationale
        output = result.specificity
        if verbose:
            print(f"The specificity of the question '{test_input["question"]["main_text"]}' is {output}.") 
            print(f"The expected specificity is {test_input["specificity"]}.")
            print(f"Rationale: {rationale}")
            print()
            print()

        expected_output = test_input["specificity"]
        if output.lower() == expected_output:
            num_correct += 1

    return num_correct

### Create AssessSpecificity Signature

In [8]:
# Create a class-based DSPy Signature to assess readability of a question

input_description = """The question to classify. The input will be a JSON with the following structure:
        {
            "response_format": "open" or "closed",
            "description": string,
            "main_text": string,
            "response_categories": empty list or list of JSONs with an "id" and "text" field
        }"""

output_description = """Whether a question is specific enough. Output "pass" if the question is specific enough, and "fail" if the question lacks sufficient specificity.""" 

class AssessSpecificity(dspy.Signature):
    """Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently)."""

    # question = dspy.InputField(desc="The question to assess.")
    question = dspy.InputField(desc=input_description)
    specificity = dspy.OutputField(desc=output_description)

#### Test AssessSpecificity Signature

In [19]:
# test out AssessBias

specificity = dspy.ChainOfThought(AssessSpecificity)

num_correct1 = test_program(specificity, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(specificity, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")


Number of correct predictions: 6/12. 50.0% accuracy.


Number of correct predictions: 2/5. 40.0% accuracy.


In [20]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

### Create AssessSpecificityModule

In [9]:
# Create a module from AssessReadability

class AssessSpecificityModule(dspy.Module):
    
    def __init__(self):

        super().__init__()

        rand_int = random.randint(1, 100)

        # define the reasoning 
        rationale_type = dspy.OutputField(
            prefix="Reasoning: Let's think step by step in order to",
            desc="${determine the specificity} We ...",
        )
        
        self.specificity = dspy.ChainOfThought(AssessSpecificity, rationale_type=rationale_type, temperature=0.7+0.0001*rand_int)

    def forward(self, question):

        # this needs to return a dict and not a string for Evaluate to work
        return self.specificity(question=question)

### Optimize AssessSpecificityModule

#### Load data and create metric

In [10]:
filename = "specificity_outputs"

# Load the training data
with open(f"generated_questions/{filename}_train.json", 'r') as f:
    train_data = json.load(f)    

# iterate through the data and construct a DSPy Example
trainset_desc = []
for item in train_data:
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_desc.append(dspy.Example(question=question_str, specificity=item["specificity"]).with_inputs("question"))   

# iterate through the data and construct a DSPy Example
trainset_no_desc = []
for item in train_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_no_desc.append(dspy.Example(question=question_str, specificity=item["specificity"]).with_inputs("question"))  

# Load the validation data
with open(f"generated_questions/{filename}_val.json", 'r') as f:
    val_data = json.load(f)

# iterate through the data and construct a DSPy Example
valset = []
for item in val_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    valset.append(dspy.Example(question=question_str, specificity=item["specificity"]).with_inputs("question")) 

print(len(trainset_desc), len(trainset_no_desc), len(valset))
print(valset[0].question)
print(trainset_desc[0].question)
print(trainset_no_desc[0].question)

144 144 48
{"response_format": "closed", "description": "", "main_text": "How satisfied are you with the city's public transportation system?", "response_categories": [{"id": 1, "text": "Very satisfied"}, {"id": 2, "text": "Somewhat satisfied"}, {"id": 3, "text": "Neutral"}, {"id": 4, "text": "Somewhat dissatisfied"}, {"id": 5, "text": "Very dissatisfied"}]}
{"response_format": "open", "description": "A question that merges different services and lacks temporal specificity.", "main_text": "Can you share experiences with public services and community events in the city?", "response_categories": []}
{"response_format": "open", "description": "", "main_text": "Can you share experiences with public services and community events in the city?", "response_categories": []}


In [11]:
# Create a metric
def validate_specificity(example, pred, trace=None):

    # make sure rationale is above a certain length
    rationale_length = len(pred.rationale.split(" "))

    # print(pred.rationale)

    if pred.specificity.lower() not in ["pass", "fail"]:
        print(f"Invalid prediction: {pred.specificity}")
        # another way of finding the category
        pred_category = ""
        if "pass" in pred.specificity.lower():
            pred_category = "pass"
        elif "fail" in pred.specificity.lower():
            pred_category = "fail"
        else:
            return False
        
        return (example.specificity.lower() == pred_category) and (rationale_length > 10)

    return (example.specificity.lower() == pred.specificity.lower()) and (rationale_length > 10)

#### Non optimized program

In [12]:
non_optimized_program = AssessSpecificityModule()

In [13]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(non_optimized_program, metric=validate_specificity)

Average Metric: 29 / 48  (60.4): 100%|██████████| 48/48 [00:19<00:00,  2.46it/s]

Average Metric: 29 / 48  (60.4%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' '✔️ [True]' '✔️ [True]' '✔️ [True]' 'False']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_specificity,rationale,pred_specificity,validate_specificity
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the city's public transportation system?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",fail,"determine the specificity. We have a single underlying concept (satisfaction with public transportation), a clear reference frame (the city's public transportation system), no ambiguous words,...",pass,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""In the past year, how often have you attended or watched a city council meeting?"", ""response_categories"": [{""id"": 1, ""text"": ""Regularly""},...",pass,"determine the specificity. 1. The question measures the frequency of attending or watching city council meetings in the past year, which is a single underlying...",pass,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How do you feel about the safety in the downtown area at night?"", ""response_categories"": []}",fail,"determine the specificity. We need to consider if the question is measuring only one underlying concept, refers to a specific reference frame (downtown area at...",fail,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How often do you participate in community events?"", ""response_categories"": [{""id"": 1, ""text"": ""Very often""}, {""id"": 2, ""text"": ""Often""}, {""id"": 3,...",fail,determine the specificity. 1. The question measures the underlying concept of participation in community events. 2. The reference frame is clear as it refers to...,fail,✔️ [True]
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How would you rate your satisfaction with the safety and cleanliness of public spaces?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""},...",fail,determine the specificity. We see that the question is measuring the concept of satisfaction with the safety and cleanliness of public spaces. It refers to...,pass,False


60.42

In [14]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

In [15]:
# Let's save the optimized programs
non_optimized_program.save('compiled_modules/assess_specificity_non_optimized.json')

In [16]:
# test on question bank

num_correct1 = test_program(non_optimized_program, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(non_optimized_program, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")

Number of correct predictions: 8/12. 66.66666666666666% accuracy.


Number of correct predictions: 1/5. 20.0% accuracy.


#### Few Shot

In [17]:
# set up optimizer 
config = dict(max_bootstrapped_demos=3, max_labeled_demos=5, max_rounds=2, max_errors=5)
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly

fewshot_optimizer = BootstrapFewShot(metric=validate_specificity, **config)
optimized_program_few_shot_no_desc = fewshot_optimizer.compile(AssessSpecificityModule(), trainset=trainset_no_desc)

  0%|          | 0/144 [00:00<?, ?it/s]

Bootstrapped 3 full traces after 1 examples in round 1.


In [18]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(optimized_program_few_shot_no_desc, metric=validate_specificity)

  0%|          | 0/48 [00:00<?, ?it/s]

Average Metric: 34 / 48  (70.8): 100%|██████████| 48/48 [00:19<00:00,  2.48it/s]

Average Metric: 34 / 48  (70.8%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' '✔️ [True]' '✔️ [True]' '✔️ [True]' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_specificity,rationale,pred_specificity,validate_specificity
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the city's public transportation system?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",fail,"determine the specificity. The question focuses on measuring the respondent's satisfaction with the city's public transportation system. It refers to a specific reference frame, which...",pass,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""In the past year, how often have you attended or watched a city council meeting?"", ""response_categories"": [{""id"": 1, ""text"": ""Regularly""},...",pass,"determine the specificity. The question focuses on one specific concept, which is the frequency of attending or watching city council meetings in the past year....",pass,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How do you feel about the safety in the downtown area at night?"", ""response_categories"": []}",fail,"determine the specificity. The question is asking for the respondent's feelings about safety in the downtown area at night, which is a specific concept. However,...",fail,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How often do you participate in community events?"", ""response_categories"": [{""id"": 1, ""text"": ""Very often""}, {""id"": 2, ""text"": ""Often""}, {""id"": 3,...",fail,"determine the specificity. The question is measuring the frequency of participation in community events, which is a specific concept. However, the response categories contain quantitative...",fail,✔️ [True]
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How would you rate your satisfaction with the safety and cleanliness of public spaces?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""},...",fail,"determine the specificity. The question is measuring the respondent's satisfaction with the safety and cleanliness of public spaces, which is a specific concept. However, the...",fail,✔️ [True]


70.83

In [19]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

In [20]:
# test on question bank

num_correct1 = test_program(optimized_program_few_shot_no_desc, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(optimized_program_few_shot_no_desc, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")

Number of correct predictions: 9/12. 75.0% accuracy.


Number of correct predictions: 2/5. 40.0% accuracy.


In [21]:
# set up optimizer 
config = dict(max_bootstrapped_demos=3, max_labeled_demos=5, max_rounds=2, max_errors=5)
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly

fewshot_optimizer = BootstrapFewShot(metric=validate_specificity, **config)
optimized_program_few_shot_desc = fewshot_optimizer.compile(AssessSpecificityModule(), trainset=trainset_desc)

  0%|          | 0/144 [00:00<?, ?it/s]

Bootstrapped 3 full traces after 1 examples in round 1.


In [22]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(optimized_program_few_shot_desc, metric=validate_specificity)

  0%|          | 0/48 [00:00<?, ?it/s]

Average Metric: 12 / 22  (54.5):  46%|████▌     | 22/48 [00:09<00:10,  2.52it/s]

Invalid prediction: partial pass


Average Metric: 32 / 48  (66.7): 100%|██████████| 48/48 [00:19<00:00,  2.44it/s]

Average Metric: 32 / 48  (66.7%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' '✔️ [True]' 'False' '✔️ [True]' 'False']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_specificity,rationale,pred_specificity,validate_specificity
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the city's public transportation system?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",fail,"determine the specificity. The question ""How satisfied are you with the city's public transportation system?"" focuses on measuring the satisfaction level of respondents with a...",pass,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""In the past year, how often have you attended or watched a city council meeting?"", ""response_categories"": [{""id"": 1, ""text"": ""Regularly""},...",pass,determine the specificity. The question specifically asks about the frequency of attending or watching city council meetings within the past year. It refers to a...,pass,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How do you feel about the safety in the downtown area at night?"", ""response_categories"": []}",fail,"determine the specificity. The question ""How do you feel about the safety in the downtown area at night?"" is specific in terms of the reference...",pass,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How often do you participate in community events?"", ""response_categories"": [{""id"": 1, ""text"": ""Very often""}, {""id"": 2, ""text"": ""Often""}, {""id"": 3,...",fail,"determine the specificity. The question ""How often do you participate in community events?"" contains quantitative adjectives like ""Very often, Often, Sometimes, Rarely, Never,"" which are...",fail,✔️ [True]
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How would you rate your satisfaction with the safety and cleanliness of public spaces?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""},...",fail,determine the specificity. The question focuses on measuring satisfaction with two specific aspects - safety and cleanliness of public spaces. It refers to a clear...,pass,False


66.67

In [23]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

In [24]:
# test on question bank

num_correct1 = test_program(optimized_program_few_shot_desc, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(optimized_program_few_shot_desc, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")

Number of correct predictions: 9/12. 75.0% accuracy.


Number of correct predictions: 1/5. 20.0% accuracy.


In [25]:
# Let's save the optimized programs
optimized_program_few_shot_no_desc.save('compiled_modules/assess_specificity_few_shot_no_desc.json')
optimized_program_few_shot_desc.save('compiled_modules/assess_specificity_few_shot_desc.json')

#### Few shot with random search

In [26]:
# Optimize the module with BootstrapFewShotWithRandomSearch
# Applies BootstrapFewShot several times with random search over generated demonstrations, and selects the best program
fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=validate_specificity, max_bootstrapped_demos=2, num_candidate_programs=8, num_threads=5)
optimized_program_few_shot_search_no_desc = fewshot_optimizer.compile(student = AssessSpecificityModule(), trainset=trainset_no_desc, valset=valset)

Going to sample between 1 and 2 traces per predictor.
Will attempt to train 8 candidate sets.


  0%|          | 0/48 [00:00<?, ?it/s]

Average Metric: 35 / 48  (72.9): 100%|██████████| 48/48 [00:19<00:00,  2.47it/s]
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)


Average Metric: 35 / 48  (72.9%)
Score: 72.92 for set: [0]
New best score: 72.92 for seed -3
Scores so far: [72.92]
Best score: 72.92


Average Metric: 30 / 48  (62.5): 100%|██████████| 48/48 [00:14<00:00,  3.39it/s]


...

Average Metric: 28 / 48  (58.3%)
Score: 58.33 for set: [16]
Scores so far: [72.92, 62.5, 60.42, 62.5, 66.67, 47.92, 60.42, 58.33, 75.0, 58.33]
Best score: 75.0
Average of max per entry across top 1 scores: 0.75
Average of max per entry across top 2 scores: 0.9583333333333334
Average of max per entry across top 3 scores: 0.9791666666666666
Average of max per entry across top 5 scores: 0.9791666666666666
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0


  2%|▏         | 3/144 [00:02<02:06,  1.11it/s]


Bootstrapped 2 full traces after 4 examples in round 0.


Average Metric: 29 / 48  (60.4): 100%|██████████| 48/48 [00:15<00:00,  3.14it/s]

Average Metric: 29 / 48  (60.4%)
Score: 60.42 for set: [16]
Scores so far: [72.92, 62.5, 60.42, 62.5, 66.67, 47.92, 60.42, 58.33, 75.0, 58.33, 60.42]
Best score: 75.0
Average of max per entry across top 1 scores: 0.75
Average of max per entry across top 2 scores: 0.9583333333333334
Average of max per entry across top 3 scores: 0.9791666666666666
Average of max per entry across top 5 scores: 0.9791666666666666
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0
11 candidate programs found.


In [27]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(optimized_program_few_shot_search_no_desc, metric=validate_specificity)

Average Metric: 36 / 48  (75.0): 100%|██████████| 48/48 [00:00<00:00, 1400.86it/s]

Average Metric: 36 / 48  (75.0%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' '✔️ [True]' 'False' '✔️ [True]' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_specificity,rationale,pred_specificity,validate_specificity
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the city's public transportation system?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",fail,determine the specificity. The question asks about the satisfaction level with the city's public transportation system. It measures a single concept (satisfaction) within a specific...,pass,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""In the past year, how often have you attended or watched a city council meeting?"", ""response_categories"": [{""id"": 1, ""text"": ""Regularly""},...",pass,determine the specificity. 1. The question measures the concept of attendance or viewership of city council meetings in a specific reference frame of the past...,pass,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How do you feel about the safety in the downtown area at night?"", ""response_categories"": []}",fail,"determine the specificity. 1. The question asks about feelings towards safety in the downtown area at night, measuring a single concept within a specific reference...",pass,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How often do you participate in community events?"", ""response_categories"": [{""id"": 1, ""text"": ""Very often""}, {""id"": 2, ""text"": ""Often""}, {""id"": 3,...",fail,determine the specificity. 1. The question measures the frequency of participation in community events. 2. The reference frame is not specific in terms of time...,fail,✔️ [True]
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How would you rate your satisfaction with the safety and cleanliness of public spaces?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""},...",fail,determine the specificity. The question asks about satisfaction with the safety and cleanliness of public spaces. It measures two concepts within a specific reference frame....,fail,✔️ [True]


75.0

In [28]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

In [29]:
# test on question bank

num_correct1 = test_program(optimized_program_few_shot_search_no_desc, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(optimized_program_few_shot_search_no_desc, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")

Number of correct predictions: 11/12. 91.66666666666666% accuracy.


Number of correct predictions: 1/5. 20.0% accuracy.


In [30]:
# Optimize the module with BootstrapFewShotWithRandomSearch
# Applies BootstrapFewShot several times with random search over generated demonstrations, and selects the best program
fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=validate_specificity, max_bootstrapped_demos=2, num_candidate_programs=8, num_threads=5)
optimized_program_few_shot_search_desc = fewshot_optimizer.compile(student = AssessSpecificityModule(), trainset=trainset_desc, valset=valset)

Going to sample between 1 and 2 traces per predictor.
Will attempt to train 8 candidate sets.


  0%|          | 0/48 [00:00<?, ?it/s]

Average Metric: 14 / 23  (60.9):  48%|████▊     | 23/48 [00:08<00:13,  1.90it/s]

Invalid prediction: pass 

Overall, the question "Is the public transportation in your city reliable and punctual?" is specific enough as it meets all the criteria for specificity.


Average Metric: 34 / 48  (70.8): 100%|██████████| 48/48 [00:16<00:00,  2.96it/s]
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)


...

Average Metric: 37 / 48  (77.1%)
Score: 77.08 for set: [16]
Scores so far: [70.83, 62.5, 72.92, 75.0, 79.17, 45.83, 68.75, 75.0, 68.75, 77.08]
Best score: 79.17
Average of max per entry across top 1 scores: 0.7916666666666666
Average of max per entry across top 2 scores: 0.9166666666666666
Average of max per entry across top 3 scores: 0.9791666666666666
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0


  1%|▏         | 2/144 [00:03<04:11,  1.77s/it]


Bootstrapped 2 full traces after 3 examples in round 0.


Average Metric: 37 / 48  (77.1): 100%|██████████| 48/48 [00:20<00:00,  2.34it/s]

Average Metric: 37 / 48  (77.1%)
Score: 77.08 for set: [16]
Scores so far: [70.83, 62.5, 72.92, 75.0, 79.17, 45.83, 68.75, 75.0, 68.75, 77.08, 77.08]
Best score: 79.17
Average of max per entry across top 1 scores: 0.7916666666666666
Average of max per entry across top 2 scores: 0.9166666666666666
Average of max per entry across top 3 scores: 0.9791666666666666
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0
11 candidate programs found.


In [31]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=1, display_progress=True, display_table=5)

evaluate_program(optimized_program_few_shot_search_desc, metric=validate_specificity)

Average Metric: 38 / 48  (79.2): 100%|██████████| 48/48 [00:00<00:00, 965.47it/s]

Average Metric: 38 / 48  (79.2%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' '✔️ [True]' 'False' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_specificity,rationale,pred_specificity,validate_specificity
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the city's public transportation system?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",fail,"determine the specificity. The question focuses on the satisfaction with the city's public transportation system. It measures one underlying concept, refers to a specific reference...",pass,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""In the past year, how often have you attended or watched a city council meeting?"", ""response_categories"": [{""id"": 1, ""text"": ""Regularly""},...",pass,"determine the specificity. We first analyze if the question measures only one underlying concept. In this case, the question focuses solely on the frequency of...",pass,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How do you feel about the safety in the downtown area at night?"", ""response_categories"": []}",fail,"determine the specificity. We should first check if the question measures only one underlying concept. In this case, the question focuses on safety in the...",pass,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How often do you participate in community events?"", ""response_categories"": [{""id"": 1, ""text"": ""Very often""}, {""id"": 2, ""text"": ""Often""}, {""id"": 3,...",fail,"determine the specificity. We first need to identify if the question measures only one underlying concept. In this case, the question asks about the frequency...",pass,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How would you rate your satisfaction with the safety and cleanliness of public spaces?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""},...",fail,"determine the specificity. We first need to check if the question measures only one underlying concept. In this case, the question combines two concepts: satisfaction...",fail,✔️ [True]


79.17

In [32]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is specific enough. Keep in mind the following qualities, for a question to be specific:
        (1) Question should measure only one underlying concept.
        (2) Question should refer to a specific reference frame (e.g. times and places) that is clear to the respondent.
        (3) Question should not contain any ambiguous words that could be interpreted in multiple ways. Question should contain terms that will have the same specific meaning to all respondents.
        (4) Question, not including the response categories, should not contain any predicates whose meanings are relative rather than absolute, as it is the case with quantitative adjectives or adverbs (e.g. often and frequently).

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list or list of JSONs with 

In [38]:
# test on question bank

num_correct1 = test_program(optimized_program_few_shot_search_desc, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

print()
print()

num_correct2 = test_program(optimized_program_few_shot_search_desc, tricky_test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct2}/{len(tricky_test_inputs)}. {num_correct2/len(tricky_test_inputs)*100}% accuracy.")

Number of correct predictions: 8/12. 66.66666666666666% accuracy.


Number of correct predictions: 2/5. 40.0% accuracy.


In [34]:
# Let's save the optimized programs
optimized_program_few_shot_search_no_desc.save('compiled_modules/assess_specificity_few_shot_search_no_desc.json')
optimized_program_few_shot_search_desc.save('compiled_modules/assess_specificity_few_shot_search_desc.json')

### Create RewriteQuestion Signature

In [35]:
# Create another class-based DSPy Signature (re-write a question based on rationale)
    
input_description = """The question to classify. The input will be a JSON with the following structure:
    {
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list or list of JSONs with an "id" and "text" field
    }"""

output_description = """The re-written question. The output will be a JSON with the following structure:
    {
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list or list of JSONs with an "id" and "text" field
    }"""
    
class RewriteQuestion(dspy.Signature):
    """Rewrite a question to address the issues identified in the input_rationale."""

    question = dspy.InputField(desc=input_description)
    input_rationale = dspy.InputField(desc="The rationale with potential issues in a question. It may contain a mix of positive and negative feedback.")
    rewritten_question = dspy.OutputField(desc=output_description)

### Create AssessSpecificityAndRewrite

In [36]:
# Create a Module with optimized_readability_program2 and RewriteQuestion

class AssessSpecificityAndRewrite(dspy.Module):
    
    def __init__(self, optimized_program):

        super().__init__()

        self.specificity = optimized_program

        self.rewrite = dspy.ChainOfThought(RewriteQuestion)

    def forward(self, question):

        result = self.specificity(question=question)

        specificity_score = result.specificity
        input_rationale = result.rationale

        assert specificity_score in ["pass", "fail"]

        if specificity_score == "fail":
            return self.rewrite(question=question, input_rationale=input_rationale).rewritten_question, input_rationale, specificity_score
        else:
            return question, input_rationale, specificity_score

In [37]:
# Test out AssessSpecificityAndRewrite

assess_specificity = AssessSpecificityModule()
assess_specificity.load('compiled_modules/assess_specificity_few_shot_search_desc.json')

# # create the module object
assess_and_rewrite = AssessSpecificityAndRewrite(assess_specificity)

num_correct = 0

# running the predictor
for test_input in test_inputs:
    # stringify the input
    test_input_str = json.dumps(test_input["question"])
    expected_output = test_input["specificity"]

    rewritten_question, rationale, score = assess_and_rewrite(question=test_input_str)

    if expected_output == score:
        num_correct += 1

    print(f"The expcted specificity is {expected_output}. The actual specificity is {score}.")
    print(f"The original question is: {test_input_str}.")
    print(f"The re-written question is: {rewritten_question}.")
    print(f"The rationale is: {rationale}.")
    print()
    print()

print(f"Number of correct predictions: {num_correct} out of {len(test_inputs)}. {num_correct/len(test_inputs)*100}% accuracy.")

The expcted specificity is fail. The actual specificity is fail.
The original question is: {"response_format": "open", "description": "", "main_text": "How big is your organization?", "response_categories": []}.
The re-written question is: {"response_format": "open", "description": "Specify the aspect of size you are referring to (e.g., number of employees, revenue, physical space).", "main_text": "How big is your organization?", "response_categories": []}.
The rationale is: determine the specificity. We first need to identify if the question measures only one underlying concept. In this case, the question asks about the size of the organization, which is a single concept. However, the question lacks specificity in terms of what aspect of size it is referring to (e.g., number of employees, revenue, physical space). Therefore, it may not be specific enough..


The expcted specificity is pass. The actual specificity is pass.
The original question is: {"response_format": "open", "descript